# Flickr30k Global Validation Lock

Tổng hợp bốn nhánh KD đã hoàn thành trên Validation, xác minh manifest và
paired-bootstrap, sau đó khóa duy nhất `T3→S3 Direct Logit-KD` làm phương
pháp cuối. Notebook chỉ cấp quyền cho một lần đánh giá Held-out Test; bản
thân notebook này không đọc Test.

## Kaggle input bắt buộc — đúng 5 input

1. Output `F30K_T1S1_SeqKD_Aggregate`.
2. Output `F30K_T2S2_SeqKD_Aggregate`.
3. Output `F30K_T1S1_DenseMap_Aggregate`.
4. Output `F30K_T3S3_LogitKD_Aggregate`.
5. Output `F30K_T3S3_Final_Lock`.

Runtime: **CPU / Accelerator None**, Internet Off, dự kiến dưới 1 phút.
Không gắn Flickr30k images, Data Contract, Teacher, checkpoint hoặc Test.
Notebook có thể chạy song song với `F30K_DenseMap_Closeout`.

## Cell 1 — Verify all Validation decisions and create lock

In [1]:
EXPERIMENT_ID = "F30K_Global_Val_Lock"


import csv
import hashlib
import json
import math
import os
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))
WORK_BASE = Path(os.environ.get("KLCN_WORK_ROOT", "/kaggle/working"))
WORK_ROOT = WORK_BASE / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

def verify_manifest(root):
    root = Path(root)
    manifest_path = root / "MANIFEST.json"
    manifest = read_json(manifest_path)
    assert manifest["status"] == "complete"
    assert manifest["test_set_status"] == "sealed"
    assert manifest["test_set_used"] is False
    for artifact in manifest["artifacts"]:
        path = root / artifact["relative_path"]
        assert path.is_file(), path
        assert path.stat().st_size == int(artifact["size_bytes"])
        assert sha256(path) == artifact["sha256"]
    return manifest_path, manifest

def write_manifest(paths, schema_version):
    artifacts = []
    for path in paths:
        path = Path(path)
        artifacts.append({
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        })
    write_json(WORK_ROOT / "MANIFEST.json", {
        "schema_version": schema_version,
        "experiment_id": EXPERIMENT_ID,
        "status": "complete",
        "artifacts": artifacts,
        "test_set_status": "sealed",
        "test_set_used": False,
    })

AGGREGATE_SPECS = [
    {
        "experiment_id": "F30K_T1S1_SeqKD_Aggregate",
        "pair_id": "T1S1_SeqKD",
        "method": "Sequence KD",
        "expected_decision": "STOP_VALIDATION_NO_GAIN",
    },
    {
        "experiment_id": "F30K_T2S2_SeqKD_Aggregate",
        "pair_id": "T2S2_SeqKD",
        "method": "Sequence KD",
        "expected_decision": "STOP_VALIDATION_NO_GAIN",
    },
    {
        "experiment_id": "F30K_T1S1_DenseMap_Aggregate",
        "pair_id": "T1S1_DenseMap",
        "method": "Dense Mapped Logit-KD",
        "expected_decision": "STOP_VALIDATION_NO_GAIN",
    },
    {
        "experiment_id": "F30K_T3S3_LogitKD_Aggregate",
        "pair_id": "T3S3_LogitKD",
        "method": "Direct Logit-KD",
        "expected_decision": "PROMOTE_TO_FINAL_METHOD_LOCK",
    },
]

registry = []
source_hashes = []
aggregates = {}
for spec in AGGREGATE_SPECS:
    result_path = unique_named(spec["experiment_id"] + "_RESULT.json")
    result = read_json(result_path)
    manifest_path, _ = verify_manifest(result_path.parent)
    assert result["status"] == "complete"
    assert result["experiment_id"] == spec["experiment_id"]
    assert result["pair_id"] == spec["pair_id"]
    assert result["method"] == spec["method"]
    assert result["seeds"] == [42, 123, 2026]
    assert int(result["run_count"]) == 3
    assert int(result["validation_images_per_run"]) == 1014
    assert result["selection_metric"] == "Validation CIDEr-D"
    assert result["decision"] == spec["expected_decision"]
    assert result["test_set_status"] == "sealed"
    assert result["test_set_used"] is False

    combined = result["combined_paired_bootstrap"]
    delta = float(result["summary"]["CIDEr_D"]["delta_mean"])
    lower = float(combined["ci95_lower"])
    upper = float(combined["ci95_upper"])
    if spec["expected_decision"] == "PROMOTE_TO_FINAL_METHOD_LOCK":
        assert result["consistent_positive_delta"] is True
        assert result["consistent_negative_delta"] is False
        assert delta > 0 and lower > 0
    else:
        assert result["consistent_positive_delta"] is False
        assert result["consistent_negative_delta"] is True
        assert delta < 0 and upper < 0
        assert result["next_phase_authorized"] is None

    registry.append({
        "pair_id": spec["pair_id"],
        "method": spec["method"],
        "baseline_CIDEr_D_mean_x100": round(
            100.0 * float(result["summary"]["CIDEr_D"]["baseline_mean"]), 6
        ),
        "kd_CIDEr_D_mean_x100": round(
            100.0 * float(result["summary"]["CIDEr_D"]["kd_mean"]), 6
        ),
        "delta_CIDEr_D_mean_x100": round(100.0 * delta, 6),
        "delta_CIDEr_D_sample_sd_x100": round(
            100.0 * float(result["summary"]["CIDEr_D"]["delta_sample_sd"]), 6
        ),
        "combined_ci95_lower_x100": round(100.0 * lower, 6),
        "combined_ci95_upper_x100": round(100.0 * upper, 6),
        "decision": result["decision"],
    })
    source_hashes.append({
        "experiment_id": spec["experiment_id"],
        "result_sha256": sha256(result_path),
        "manifest_sha256": sha256(manifest_path),
    })
    aggregates[spec["pair_id"]] = result

positive_rows = [
    row for row in registry
    if row["decision"] == "PROMOTE_TO_FINAL_METHOD_LOCK"
]
assert len(positive_rows) == 1
assert positive_rows[0]["pair_id"] == "T3S3_LogitKD"

FINAL_LOCK_ID = "F30K_T3S3_Final_Lock"
final_lock_path = unique_named(FINAL_LOCK_ID + "_RESULT.json")
final_lock = read_json(final_lock_path)
final_lock_manifest_path, _ = verify_manifest(final_lock_path.parent)
assert final_lock["status"] == "locked"
assert final_lock["experiment_id"] == FINAL_LOCK_ID
assert final_lock["method"] == "Direct Logit-KD"
assert final_lock["test_set_status"] == "sealed"
assert final_lock["test_set_used"] is False
selected = final_lock["deployment_checkpoint_selection"]
assert int(selected["seed"]) == 42
assert selected["experiment_id"] == "F30K_T3S3_LogitKD_42"
assert selected["best_checkpoint_sha256"] == (
    "862f16b5b08bf580eac13ac112d529e9beeac7b8420116c598210f5b2c3ea4b4"
)

winning_aggregate = aggregates["T3S3_LogitKD"]
assert math.isclose(
    float(selected["validation_CIDEr_D"]),
    max(float(row["kd_CIDEr_D"]) for row in winning_aggregate["runs"]),
    rel_tol=0.0,
    abs_tol=1e-12,
)

global_lock = {
    "schema_version": "f30k_global_validation_lock_v1",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "locked",
    "experiment_id": EXPERIMENT_ID,
    "dataset": "Flickr30k",
    "evaluation_boundary": "Validation-only method selection",
    "metric_display_scale": 100,
    "candidate_registry": registry,
    "selected_method": {
        "pair_id": "T3S3_LogitKD",
        "teacher": "T3 SigLIP ViT-B/16 adapted on Flickr30k",
        "student": "S3 TinyViT-5M",
        "method": "Direct Logit-KD",
        "scientific_seed_count": 3,
        "mean_delta_CIDEr_D_x100": round(
            100.0 * float(
                winning_aggregate["summary"]["CIDEr_D"]["delta_mean"]
            ),
            6,
        ),
        "combined_paired_bootstrap_ci95_x100": [
            round(
                100.0 * float(
                    winning_aggregate["combined_paired_bootstrap"]["ci95_lower"]
                ),
                6,
            ),
            round(
                100.0 * float(
                    winning_aggregate["combined_paired_bootstrap"]["ci95_upper"]
                ),
                6,
            ),
        ],
        "deployment_seed": 42,
        "deployment_experiment_id": selected["experiment_id"],
        "deployment_best_epoch": int(selected["best_epoch"]),
        "deployment_validation_CIDEr_D_x100": round(
            100.0 * float(selected["validation_CIDEr_D"]), 6
        ),
        "deployment_checkpoint_sha256": selected["best_checkpoint_sha256"],
        "deployment_checkpoint_bytes": int(selected["best_checkpoint_bytes"]),
    },
    "selection_policy": {
        "primary_metric": "Validation CIDEr-D",
        "uncertainty": "paired bootstrap by image ID, 10000 replicates",
        "scientific_estimate": "mean and sample SD over seeds 42, 123, 2026",
        "deployment_checkpoint": "highest Validation CIDEr-D only",
        "method_or_threshold_changes_after_lock": False,
    },
    "source_hashes": source_hashes + [{
        "experiment_id": FINAL_LOCK_ID,
        "result_sha256": sha256(final_lock_path),
        "manifest_sha256": sha256(final_lock_manifest_path),
    }],
    "held_out_test_authorized": True,
    "held_out_test_authorization_count": 1,
    "held_out_test_scope": (
        "Evaluate only the locked T3S3 Direct Logit-KD seed-42 checkpoint "
        "once; no reselection, retraining, threshold change, or method change."
    ),
    "next_phase_authorized": "F30K_T3S3_HeldOut_Test_OneTime",
    "test_set_status": "sealed_until_next_notebook",
    "test_set_used": False,
}

lock_path = WORK_ROOT / "F30K_GLOBAL_VALIDATION_LOCK.json"
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
csv_path = WORK_ROOT / "validation_method_registry_x100.csv"
report_path = WORK_ROOT / "REPORT.md"
write_json(lock_path, global_lock)
write_json(result_path, global_lock)
with csv_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(registry[0]))
    writer.writeheader()
    writer.writerows(registry)

lines = [
    "# Flickr30k global Validation lock",
    "",
    "| Pair/method | Baseline CIDEr-D | KD CIDEr-D | Δ CIDEr-D | CI95 | Decision |",
    "|---|---:|---:|---:|---:|---|",
]
for row in registry:
    lines.append(
        f"| {row['pair_id']} / {row['method']} | "
        f"{row['baseline_CIDEr_D_mean_x100']:.2f} | "
        f"{row['kd_CIDEr_D_mean_x100']:.2f} | "
        f"{row['delta_CIDEr_D_mean_x100']:+.2f} | "
        f"[{row['combined_ci95_lower_x100']:+.2f}; "
        f"{row['combined_ci95_upper_x100']:+.2f}] | "
        f"`{row['decision']}` |"
    )
lines.extend([
    "",
    "- Selected method: `T3→S3 Direct Logit-KD`.",
    "- Deployment checkpoint: seed `42`, selected using Validation only.",
    "- Held-out Test: authorized exactly once in the next notebook.",
    "- No method, checkpoint, threshold or hyperparameter may be changed "
    "after opening Test.",
])
report_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
write_manifest(
    [lock_path, result_path, csv_path, report_path],
    "f30k_global_validation_lock_manifest_v1",
)

print("Status                 : LOCKED")
print("Candidates verified    : 4 / 4")
print("Selected pair/method   : T3S3 / Direct Logit-KD")
print("Deployment seed        : 42")
print("Checkpoint SHA-256     :", selected["best_checkpoint_sha256"])
print("Held-out Test used     : False")
print("Held-out Test next     : AUTHORIZED EXACTLY ONCE")
print("Next                   :", global_lock["next_phase_authorized"])

Status                 : LOCKED
Candidates verified    : 4 / 4
Selected pair/method   : T3S3 / Direct Logit-KD
Deployment seed        : 42
Checkpoint SHA-256     : 862f16b5b08bf580eac13ac112d529e9beeac7b8420116c598210f5b2c3ea4b4
Held-out Test used     : False
Held-out Test next     : AUTHORIZED EXACTLY ONCE
Next                   : F30K_T3S3_HeldOut_Test_OneTime
